# Chatbot V1

In [ ]:
# with open(".env", "w") as f:
#     f.write("OPENAI_API_KEY=YOUR_KEY_GOES_HERE\n")

In [8]:
import os
import json
from typing import Dict, List, Any, Optional, Union
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.schema import OutputParserException
from langchain_core.output_parsers import StrOutputParser
import time
import random
from dotenv import load_dotenv
load_dotenv()  # Loads variables from .env
api_key = os.getenv("OPENAI_API_KEY")


class PropertyRequirements(BaseModel):
    """Structured data model for property requirements"""
    purpose: Optional[str] = Field(None, description="Purpose of search: 'buy' or 'rent'")
    property_type: Optional[str] = Field(None, description="Type of property: chalet, countryHouse, duplex, flat, penthouse, studio")
    country: Optional[str] = Field(None, description="Country code ")
    municipality: Optional[str] = Field(None, description="Municipality or city")
    price: Optional[str] = Field(None, description="Price range: 100K-300K, 300K-600K, 600K-1M, 1M+")
    currency: Optional[str] = Field(None, description="Currency (default: euro)")
    rooms: Optional[str] = Field(None, description="Number of bedrooms")
    bathrooms: Optional[str] = Field(None, description="Number of bathrooms")
    size: Optional[str] = Field(None, description="Size in square meters")
    status: Optional[str] = Field(None, description="Property status: good, newdevelopment, renew")
    amenities: Optional[List[str]] = Field(None, description="Desired amenities")
    floor: Optional[str] = Field(None, description="Floor number or level")


class RealEstateLangchainChatbot:
    def __init__(self, api_key: str = None):
        """
        Initialize the chatbot with OpenAI API key
        
        Args:
            api_key: OpenAI API key (if None, will try to use environment variable)
        """
        # Set OpenAI API key
        if api_key:
            os.environ["OPENAI_API_KEY"] = api_key
        elif not os.getenv("OPENAI_API_KEY"):
            raise ValueError("API key must be provided or set as OPENAI_API_KEY environment variable")
        
        # Initialize LLM
        self.extraction_llm = ChatOpenAI(
            model="gpt-4-turbo",
            temperature=0.1,
        )
        
        self.response_llm = ChatOpenAI(
            model="gpt-3.5-turbo",
            temperature=0.7,
        )
        
        # Initialize Pydantic parser for structured output
        self.property_parser = PydanticOutputParser(pydantic_object=PropertyRequirements)
        
        # Initialize conversation memory
        self.memory = ConversationBufferMemory(return_messages=True)
        
        # Required information slots
        self.required_slots = [
            "purpose",
            "property_type",
            "municipality",
            "price",
            "rooms"
        ]
        
        # Additional preferences to ask about after required info
        self.additional_slots = [
            "bathrooms",
            "size",
            "amenities",
            "floor",
            "status"
        ]
        
        # Common amenities for the Spanish market
        self.common_amenities = [
            "garden", "terrace", "balcony", "private pool", 
            "air conditioning", "heating", "elevator", "garage", 
            "parking", "mountain views", "storage room"
        ]
        
        # Initialize user requirements
        self.user_requirements = PropertyRequirements()
        
        # Track conversation state
        # possible states: greeting, collecting_required, confirming_required, 
        # collecting_additional, confirming_all, completed
        self.state = "greeting"
        
        # Track which additional preferences we've asked about
        self.asked_additional = []
        
        # Starting message
        self.starting_message = (
            "Hello! I'm your real estate assistant. Please tell me about the property you're looking for. "
            "Feel free to include details like:\n\n"
            "• Are you looking to buy or rent?\n"
            "• What type of property (flat, chalet, penthouse, studio)?\n"
            "• Which municipality or city?\n"
            "• Your budget range?\n"
            "• How many bedrooms do you need?\n"
            "• Any special features or amenities? (terrace, garden, pool, etc.)\n\n"
            "Just describe what you're looking for in your own words, and I'll gather the details."
        )

    def extract_property_info(self, user_message: str) -> PropertyRequirements:
        """
        Extract property information from user input using LangChain
        
        Args:
            user_message: The message from the user
            
        Returns:
            PropertyRequirements object with extracted information
        """
        # Create extraction prompt
        extraction_prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content=f"""
                You are an AI assistant specializing in real estate. Extract property requirements from the user's message.
                Extract ONLY the information explicitly mentioned by the user. Do not assume or infer information not stated.
                
                Pay special attention to:
                1. Whether they want to buy or rent (purpose)
                2. Property type (flat, chalet, countryHouse, duplex, penthouse, studio)
                3. Municipality or city
                4. Price/budget ranges (in Euros by default)
                5. Number of bedrooms (rooms) and bathrooms
                6. AMENITIES like terrace, garden, pool, etc. - be thorough with these
                
                For amenities, be especially vigilant. Look for mentions of:
                {', '.join(self.common_amenities)}
                
                Preserve ranges when mentioned (like 4-7 bedrooms or 500K-700K).
                
                Format your response as a valid JSON that matches the PropertyRequirements schema.
            """),
            MessagesPlaceholder(variable_name="history"),
            HumanMessage(content=f"Extract property requirements from this message: '{user_message}'"),
            SystemMessage(content=f"Return a JSON object that follows this format: {self.property_parser.get_format_instructions()}")
        ])
        
        # Create chain for extraction
        extraction_chain = (
            extraction_prompt 
            | self.extraction_llm
            | StrOutputParser()
        )
        
        try:
            # Run extraction chain
            result = extraction_chain.invoke({"history": self.memory.load_memory_variables({})["history"]})
            
            # Parse the result into a PropertyRequirements object
            try:
                # Try to parse as JSON first
                json_result = json.loads(result)
                # Convert to PropertyRequirements
                extracted_info = PropertyRequirements(**json_result)
                return extracted_info
            except json.JSONDecodeError:
                # If JSON parsing fails, try using the pydantic parser directly
                return self.property_parser.parse(result)
            
        except OutputParserException as e:
            print(f"Error parsing output: {e}")
            return PropertyRequirements()
        except Exception as e:
            print(f"Error extracting property information: {e}")
            return PropertyRequirements()
            
    def extract_amenities_from_message(self, user_message: str) -> List[str]:
        """
        Directly extract amenities from a user message using keyword matching
        
        Args:
            user_message: The message from the user
            
        Returns:
            List of amenities found in the message
        """
        found_amenities = []
        lower_message = user_message.lower()
        
        # Check for common amenities using simple keyword matching
        for amenity in self.common_amenities:
            if amenity in lower_message:
                found_amenities.append(amenity)
                
        # Apply basic cleaning and deduplication
        cleaned_amenities = []
        seen = set()
        
        for amenity in found_amenities:
            # Handle similar terms (e.g., "pool" and "private pool")
            if "pool" in amenity and any(pool_term in seen for pool_term in ["pool", "private pool"]):
                continue
                
            seen.add(amenity)
            cleaned_amenities.append(amenity)
            
        return cleaned_amenities

    def update_requirements(self, extracted_info: PropertyRequirements):
        """Update the stored requirements with new extracted information"""
        # Convert current requirements to dict
        current_dict = self.user_requirements.model_dump()
        
        # Convert extracted info to dict, filtering out None values
        extracted_dict = {k: v for k, v in extracted_info.model_dump().items() if v is not None and v != []}
        
        # Special handling for amenities to avoid overwriting
        if "amenities" in extracted_dict and current_dict.get("amenities"):
            # Merge the amenities lists without duplicates
            current_amenities = set(current_dict["amenities"])
            new_amenities = set(extracted_dict["amenities"])
            merged_amenities = list(current_amenities.union(new_amenities))
            extracted_dict["amenities"] = merged_amenities
        
        # Update current dict with extracted dict
        current_dict.update(extracted_dict)
        
        # Create a new PropertyRequirements object
        self.user_requirements = PropertyRequirements(**current_dict)
        
        # Perform additional direct amenity extraction as a fallback
        amenities_from_keywords = self.extract_amenities_from_message(self.memory.chat_memory.messages[-1].content)
        if amenities_from_keywords:
            current_amenities = self.user_requirements.amenities or []
            # Add any amenities found by keyword matching that weren't already captured
            for amenity in amenities_from_keywords:
                if amenity not in current_amenities:
                    current_amenities.append(amenity)
            self.user_requirements.amenities = current_amenities
    
    def get_missing_required_slots(self) -> List[str]:
        """Return a list of required slots that are still missing"""
        requirements_dict = self.user_requirements.model_dump()
        return [slot for slot in self.required_slots if requirements_dict.get(slot) is None]
    
    def generate_bulk_question(self) -> str:
        """Generate a question asking for multiple missing required fields at once"""
        missing_slots = self.get_missing_required_slots()
        
        if not missing_slots:
            return self.generate_confirmation_message()
        
        # Create a more natural list of what we're asking about
        friendly_names = {
            "purpose": "whether you want to buy or rent",
            "property_type": "what type of property you're looking for (flat, chalet, penthouse, etc.)",
            "municipality": "which municipality or city you're interested in",
            "price": "your budget range in euros",
            "rooms": "how many bedrooms you need"
        }
        
        missing_friendly = [friendly_names[slot] for slot in missing_slots]
        
        # Format the list properly with Oxford comma
        if len(missing_friendly) == 1:
            missing_text = missing_friendly[0]
        elif len(missing_friendly) == 2:
            missing_text = f"{missing_friendly[0]} and {missing_friendly[1]}"
        else:
            missing_text = ", ".join(missing_friendly[:-1]) + f", and {missing_friendly[-1]}"
        
        # Get requirements as dict for template
        requirements_dict = self.user_requirements.model_dump()
        
        # Filter out None values for display
        filled_requirements = {k: v for k, v in requirements_dict.items() if v is not None}
        
        # Create question prompt
        question_prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content=f"""
                You are a friendly real estate assistant. Ask for the following information in a single question:
                {missing_text}.
                
                Important guidelines:
                1. Keep your question concise and conversational
                2. Acknowledge what you already know about their requirements
                3. Address the user directly 
                4. Phrase it as one cohesive question, not a list of separate questions
                5. Make it easy for them to respond naturally
            """),
            SystemMessage(content=f"""
                Current user requirements: {json.dumps(filled_requirements, indent=2)}
                
                Ask about the missing information in one natural question.
            """)
        ])
        
        # Create chain for question generation
        question_chain = question_prompt | self.response_llm | StrOutputParser()
        
        try:
            # Run question chain
            return question_chain.invoke({})
        except Exception as e:
            print(f"Error generating question: {e}")
            # Fallback question
            return f"Could you please tell me {missing_text}?"
    
    def generate_confirmation_message(self, final=False) -> str:
        """Generate a confirmation message summarizing the collected information"""
        # Get requirements as dict for template
        requirements_dict = self.user_requirements.model_dump()
        
        # Filter out None values for display
        filled_requirements = {k: v for k, v in requirements_dict.items() if v is not None}
        
        if final:
            # Create final confirmation prompt with all details
            confirmation_prompt = ChatPromptTemplate.from_messages([
                SystemMessage(content="""
                    You are a professional real estate assistant summarizing the requirements shared by the user.
                    
                    Important guidelines:
                    1. Address the user directly ("you are looking for" not "the client is looking for")
                    2. Summarize ALL requirements in natural language paragraphs
                    3. BE SURE to include ALL amenities the user mentioned
                    4. List amenities clearly - never omit terraces, gardens or other key features
                    5. Preserve ranges when mentioned (e.g., "4-7 bedrooms" not just "4 bedrooms")
                    6. End by asking if the summary is correct and if they'd like to add any other important details
                """),
                SystemMessage(content=f"""
                    User requirements: {json.dumps(filled_requirements, indent=2)}
                    
                    Summarize ALL these requirements in natural language and ask for confirmation.
                    Make sure to highlight all amenities clearly in your summary.
                """)
            ])
        else:
            # Create initial confirmation prompt with just the required details
            confirmation_prompt = ChatPromptTemplate.from_messages([
                SystemMessage(content="""
                    You are a professional real estate assistant summarizing the requirements shared by the user.
                    
                    Important guidelines:
                    1. Address the user directly ("you are looking for" not "the client is looking for")
                    2. Summarize their basic requirements in a single paragraph
                    3. ALWAYS mention amenities if they're provided - NEVER omit them
                    4. Be particularly careful to mention terraces, gardens and private pools if specified
                    5. Preserve ranges when mentioned (e.g., "4-7 bedrooms" not just "4 bedrooms")
                    6. End by asking if the basic details are correct
                    7. Keep it brief but complete
                """),
                SystemMessage(content=f"""
                    User requirements: {json.dumps(filled_requirements, indent=2)}
                    
                    Summarize the basic requirements and ask for confirmation.
                    REMEMBER to include amenities in your summary if they're provided.
                """)
            ])
        
        # Create chain for confirmation generation
        confirmation_chain = confirmation_prompt | self.response_llm | StrOutputParser()
        
        try:
            # Run confirmation chain
            return confirmation_chain.invoke({})
        except Exception as e:
            print(f"Error generating confirmation: {e}")
            # Fallback confirmation
            if final:
                properties_list = [f"• {k.replace('_', ' ')}: {v}" for k, v in filled_requirements.items()]
                properties_text = "\n".join(properties_list)
                return f"Here's a summary of all your requirements:\n\n{properties_text}\n\nIs this information correct? Would you like to add any other important details?"
            else:
                basic_reqs = {k: v for k, v in filled_requirements.items() if k in self.required_slots}
                amenities = filled_requirements.get("amenities", [])
                if amenities:
                    amenities_text = ", ".join(amenities)
                    properties_text = ", ".join([f"{k.replace('_', ' ')}: {v}" for k, v in basic_reqs.items()])
                    return f"You're looking for a property with these details: {properties_text}, with amenities including {amenities_text}. Is that correct?"
                else:
                    properties_text = ", ".join([f"{k.replace('_', ' ')}: {v}" for k, v in basic_reqs.items()])
                    return f"You're looking for a property with these details: {properties_text}. Is that correct?"
    
    def get_additional_questions_batch(self) -> str:
        """Generate a question asking about multiple additional preferences at once"""
        # Get available additional slots to ask about (up to 3 at a time)
        available_slots = [slot for slot in self.additional_slots if slot not in self.asked_additional]
        batch_slots = available_slots[:3]
        
        if not batch_slots:
            # If we've asked about all additional preferences
            return self.generate_confirmation_message(final=True)
        
        # Mark these slots as asked
        for slot in batch_slots:
            self.asked_additional.append(slot)
        
        # Get requirements as dict
        requirements_dict = self.user_requirements.model_dump()
        
        # Filter out None values for display
        filled_requirements = {k: v for k, v in requirements_dict.items() if v is not None}
        
        # Create friendly names for the slots
        friendly_names = {
            "bathrooms": "number of bathrooms",
            "size": "property size (in square meters)",
            "amenities": "desired amenities (terrace, garden, elevator, etc.)",
            "floor": "which floor level you prefer",
            "status": "property status (good condition, new development, renovated)"
        }
        
        # Format the list of what we're asking about
        batch_friendly = [friendly_names[slot] for slot in batch_slots]
        if len(batch_friendly) == 1:
            batch_text = batch_friendly[0]
        elif len(batch_friendly) == 2:
            batch_text = f"{batch_friendly[0]} and {batch_friendly[1]}"
        else:
            batch_text = ", ".join(batch_friendly[:-1]) + f", and {batch_friendly[-1]}"
        
        # Create question prompt
        question_prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content=f"""
                You are a friendly real estate assistant asking about additional preferences.
                You've already collected the essential information, and now you're asking about: {batch_text}
                
                Important guidelines:
                1. Make it clear these are additional preferences (not mandatory)
                2. Ask about all these preferences in ONE concise question
                3. Make it easy for the user to skip any or all of them
                4. Be conversational and friendly
                5. Tailor your question based on what you already know about their requirements
                6. If they already mentioned some amenities, ask if they want OTHER amenities
            """),
            SystemMessage(content=f"""
                Current user requirements: {json.dumps(filled_requirements, indent=2)}
                
                Ask about the additional preferences in one natural question.
                If amenities are already specified, ask if they want any OTHER amenities.
            """)
        ])
        
        # Create chain for question generation
        question_chain = question_prompt | self.response_llm | StrOutputParser()
        
        try:
            # Run question chain
            return question_chain.invoke({})
        except Exception as e:
            print(f"Error generating additional questions: {e}")
            # Fallback question
            return f"Could you share any preferences about {batch_text}? (Feel free to skip any that aren't important to you.)"
    
    def double_check_for_amenities(self) -> bool:
        """Check if amenities need to be explicitly verified before completing"""
        # If we're in the confirming_all state and haven't specifically asked about amenities
        if self.state == "confirming_all" and "amenities" not in self.asked_additional:
            # Check if there are any amenities in the history that might not have been captured
            history = self.memory.load_memory_variables({})["history"]
            
            # Analyze last few messages for potential amenities
            potential_amenities = []
            for msg in history[-3:]:  # Check last 3 messages
                for amenity in self.common_amenities:
                    if amenity in msg.content.lower() and amenity not in potential_amenities:
                        potential_amenities.append(amenity)
            
            # If we found potential amenities and they're not in the current requirements
            current_amenities = self.user_requirements.amenities or []
            missing_amenities = [a for a in potential_amenities if a not in current_amenities]
            
            return len(missing_amenities) > 0
        
        return False
        
    def get_structured_data(self) -> Dict[str, Any]:
        """Return the collected data in a structured format"""
        # Add default country if not specified
        requirements_dict = self.user_requirements.model_dump()
        if requirements_dict.get("country") is None:
            requirements_dict["country"] = "es"
        
        # Add default currency if not specified
        if requirements_dict.get("currency") is None and requirements_dict.get("price") is not None:
            requirements_dict["currency"] = "€"
            
        return {k: v for k, v in requirements_dict.items() if v is not None}
    
    def prepare_response_for_display(self, response):
        """Format final output for display"""
        final_response = f"{response}\n\nSearch data: {json.dumps(self.get_structured_data(), indent=2)}"
        return final_response
    
    def process_message(self, user_message: str) -> str:
        """
        Process a user message and return a response
        
        Args:
            user_message: The message from the user
            
        Returns:
            A response message from the chatbot
        """
        # Add user message to memory
        self.memory.chat_memory.add_user_message(user_message)
        
        # Extract amenities directly from the message for backup
        direct_amenities = self.extract_amenities_from_message(user_message)
        if direct_amenities and "terrace" in " ".join(direct_amenities).lower():
            # If we detect the word "terrace" in this message, make sure it's captured
            print(f"Direct amenity detection found: {direct_amenities}")
        
        # Handle empty first message
        if not user_message.strip():
            if self.state == "greeting":
                self.memory.chat_memory.add_ai_message(self.starting_message)
                return self.starting_message
        
        # Handle greeting state
        if self.state == "greeting":
            # Extract any information from the greeting
            extracted_info = self.extract_property_info(user_message)
            self.update_requirements(extracted_info)
            
            # Check if we have all required information from the initial message
            missing_slots = self.get_missing_required_slots()
            
            if not missing_slots:
                # If we have all required info, move to confirmation
                self.state = "confirming_required"
                response = self.generate_confirmation_message()
                self.memory.chat_memory.add_ai_message(response)
                return response
            elif len(missing_slots) <= 2:
                # If we're only missing 1-2 fields, ask specifically for those
                self.state = "collecting_required"
                response = self.generate_bulk_question()
                self.memory.chat_memory.add_ai_message(response)
                return response
            else:
                # If we're missing several fields, ask for all of them at once
                self.state = "collecting_required"
                response = ("Thanks for that information! To help me find the perfect property for you, "
                          "could you please tell me:\n\n" + 
                          ", ".join(f"{slot.replace('_', ' ')}" for slot in missing_slots) + "?")
                self.memory.chat_memory.add_ai_message(response)
                return response
        
        # Handle collecting required info state
        elif self.state == "collecting_required":
            # Extract information from user message
            extracted_info = self.extract_property_info(user_message)
            self.update_requirements(extracted_info)
            
            # Check if we have all required information
            missing_slots = self.get_missing_required_slots()
            
            if not missing_slots:
                self.state = "confirming_required"
                response = self.generate_confirmation_message()
                self.memory.chat_memory.add_ai_message(response)
                return response
            else:
                # If still need more required information, ask for all missing fields at once
                response = self.generate_bulk_question()
                self.memory.chat_memory.add_ai_message(response)
                return response
        
        # Handle confirming required info state
        elif self.state == "confirming_required":
            # Check if user confirmed basic requirements
            if any(word in user_message.lower() for word in ["yes", "correct", "right", "sure", "ok", "looks good", "yep", "yeah", "sí", "si"]):
                # Start asking about additional preferences
                self.state = "collecting_additional"
                response = self.get_additional_questions_batch()
                self.memory.chat_memory.add_ai_message(response)
                return response
            else:
                # Go back to collecting required info if confirmation failed
                self.state = "collecting_required"
                # Extract any corrections from the message
                extracted_info = self.extract_property_info(user_message)
                self.update_requirements(extracted_info)
                
                # Check if we have all required information after corrections
                if not self.get_missing_required_slots():
                    self.state = "confirming_required"
                    response = self.generate_confirmation_message()
                    self.memory.chat_memory.add_ai_message(response)
                    return response
                else:
                    response = "Let me update that. What other details would you like to provide about your requirements?"
                    self.memory.chat_memory.add_ai_message(response)
                    return response
        
        # Handle collecting additional info state
        elif self.state == "collecting_additional":
            # Extract information from user message
            extracted_info = self.extract_property_info(user_message)
            self.update_requirements(extracted_info)
            
            # Check if we've asked all additional questions
            if len(self.asked_additional) >= len(self.additional_slots) or "skip" in user_message.lower():
                # Move to final confirmation
                self.state = "confirming_all"
                response = self.generate_confirmation_message(final=True)
                self.memory.chat_memory.add_ai_message(response)
                return response
            else:
                # Ask about the next batch of additional preferences
                response = self.get_additional_questions_batch()
                self.memory.chat_memory.add_ai_message(response)
                return response
        
        # Handle confirming all info state
        elif self.state == "confirming_all":
            # First extract any final details from the message
            extracted_info = self.extract_property_info(user_message)
            self.update_requirements(extracted_info)
            
            # Check if we need to specifically verify amenities (like terraces/gardens)
            for key_amenity in ["terrace", "garden", "private pool"]:
                if key_amenity in user_message.lower() and (not self.user_requirements.amenities or key_amenity not in " ".join(self.user_requirements.amenities).lower()):
                    # Make sure the amenity gets added if it's mentioned
                    current_amenities = self.user_requirements.amenities or []
                    current_amenities.append(key_amenity)
                    self.user_requirements.amenities = current_amenities
                    
                    # Generate updated confirmation
                    response = f"I've added the {key_amenity} to your requirements. Here's the updated summary:\n\n"
                    response += self.generate_confirmation_message(final=True)
                    self.memory.chat_memory.add_ai_message(response)
                    return response
            
            # Check if user confirmed all information
            if any(word in user_message.lower() for word in ["yes", "correct", "right", "sure", "ok", "looks good", "yep", "yeah"]):
                # Final check for any new amenities or requirements
                if self.double_check_for_amenities():
                    # Ask one final verification about amenities
                    response = "Just to double-check - are there any specific amenities like a terrace, garden, private pool, or other features that you'd like the property to have?"
                    self.memory.chat_memory.add_ai_message(response)
                    
                    # Create a special state for this final check
                    self.state = "final_amenities_check"
                    return response
                
                # Complete the conversation
                self.state = "completed"
                response = "Perfect! I've got all your requirements. I'll find some great properties that match your criteria. Thank you for providing this information!"
                self.memory.chat_memory.add_ai_message(response)
                return self.prepare_response_for_display(response)
            else:
                # Extract any corrections from the message
                extracted_info = self.extract_property_info(user_message)
                self.update_requirements(extracted_info)
                
                # Generate updated confirmation
                response = "I've updated your preferences. Here's the final summary:\n\n"
                response += self.generate_confirmation_message(final=True)
                self.memory.chat_memory.add_ai_message(response)
                return response
        
        # Handle final amenities check state
        elif self.state == "final_amenities_check":
            # Extract any final amenities
            extracted_info = self.extract_property_info(user_message)
            self.update_requirements(extracted_info)
            
            # Move to completed state
            self.state = "completed"
            response = "Perfect! I've got all your requirements, including your amenity preferences. I'll find some great properties that match your criteria. Thank you for providing this information!"
            self.memory.chat_memory.add_ai_message(response)
            return self.prepare_response_for_display(response)
                
        # Default case (shouldn't normally reach here)
        return "I'm not sure what happened. Let's start again. What type of property are you looking for?"



def run_chatbot_interactive():
    """Run the chatbot interactively"""
    api_key = input("Enter your OpenAI API key (or press Enter to use OPENAI_API_KEY environment variable): ")
    if not api_key:
        api_key = None
    
    chatbot = RealEstateLangchainChatbot(api_key=api_key)
    print("Real Estate Chatbot is ready! Type 'exit' to quit.")
    
    # Display the initial greeting
    print(f"Bot: {chatbot.starting_message}")
    
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            break
            
        response = chatbot.process_message(user_input)
        print("Bot:", response)
        
        # If we have completed the information gathering
        if chatbot.state == "completed":
            break

if __name__ == "__main__":
    run_chatbot_interactive()

Enter your OpenAI API key (or press Enter to use OPENAI_API_KEY environment variable):  


Real Estate Chatbot is ready! Type 'exit' to quit.
Bot: Hello! I'm your real estate assistant. Please tell me about the property you're looking for. Feel free to include details like:

• Are you looking to buy or rent?
• What type of property (flat, chalet, penthouse, studio)?
• Which municipality or city?
• Your budget range?
• How many bedrooms do you need?
• Any special features or amenities? (terrace, garden, pool, etc.)

Just describe what you're looking for in your own words, and I'll gather the details.


You:  i want a flat in Madrid for rent


Bot: What's your budget range in euros and how many bedrooms are you looking for in a flat to rent in Madrid?


You:  up to 1000 per month, 1-2 bedrooms


...

Bot: That's great! Just to make sure we find the perfect place for you, do you have any preferences regarding the floor level or the property status? And are there any other specific amenities you'd like to add to your list? Feel free to share any additional details you have in mind!


You:  terrace, elevator


Direct amenity detection found: ['terrace', 'elevator']
Bot: You are looking to rent a flat in Madrid for up to €1000 per month. You specifically need a property with 1-2 rooms and 1 bathroom. Additionally, you require the flat to have a terrace and an elevator.

Does this summary correctly capture all your requirements, or is there anything else you would like to add or modify?


You:  yes


Bot: Perfect! I've got all your requirements. I'll find some great properties that match your criteria. Thank you for providing this information!

Search data: {
  "purpose": "rent",
  "property_type": "flat",
  "country": "es",
  "municipality": "Madrid",
  "price": "up to 1000",
  "currency": "\u20ac",
  "rooms": "1-2",
  "bathrooms": "1",
  "amenities": [
    "terrace",
    "elevator"
  ]
}
